# Lab4-2. Improving MLP Regression: ReLU and Adam

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GLI-Lab/machine-learning-course/blob/students/exercises/lab04/mlp-regression-scratch2.ipynb)

## Objectives

- Observe how replacing `Sigmoid` with **ReLU** in hidden layers mitigates the vanishing gradient problem and improves convergence.
- Observe how replacing the SGD `Optimizer` with **Adam** dramatically accelerates training, achieving better results in far fewer epochs.

> #### 📝 Building on Lab 3-4
>
> This lab reuses the same `Linear`, `Identity`, `MSELoss`, `MLP`, and `train()` infrastructure from Lab 3-4. We only add two new components: a `ReLU` activation class and an `Adam` optimizer class. Everything else remains unchanged.

## 0. Setup and Data Preparation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.2)

# ============================================================
# Data Preparation (identical to Lab 3-4)
# ============================================================
data = np.genfromtxt("../../data/london_houses_transformed.csv", delimiter=",", skip_header=1)
X = data[:, :-1]
Y = data[:, -1:]

def train_test_split(X, Y, test_size=0.2, random_state=15):
    np.random.seed(random_state)
    N = X.shape[0]
    indices = np.random.permutation(N)
    split = int(N * (1 - test_size))
    train_idx, test_idx = indices[:split], indices[split:]
    return X[train_idx], X[test_idx], Y[train_idx], Y[test_idx]

X_train_raw, X_test_raw, Y_train_raw, Y_test_raw = train_test_split(
    X, Y, test_size=0.2, random_state=15
)

X_mean, X_std = X_train_raw.mean(axis=0), X_train_raw.std(axis=0) + 1e-8
X_train = (X_train_raw - X_mean) / X_std
X_test  = (X_test_raw  - X_mean) / X_std

Y_mean, Y_std = Y_train_raw.mean(), Y_train_raw.std()
Y_train = (Y_train_raw - Y_mean) / Y_std
Y_test  = (Y_test_raw  - Y_mean) / Y_std

N, D = X_train.shape
print(f"Dataset: {X.shape[0]} total samples")
print(f"  Train: {N} | Test: {X_test.shape[0]}")

## 1. MLP Regression from Scratch (Sigmoid + SGD)

In [ ]:
class Linear:
    def __init__(self, in_dims, out_dims):
        bound = 1 / np.sqrt(in_dims)
        self.W = np.random.uniform(-bound, bound, size=(out_dims, in_dims))
        self.b = np.random.uniform(-bound, bound, size=(out_dims,))

    def forward(self, x):
        self.x = x
        return x @ self.W.T + self.b

    def backward(self, grad):
        self.deltaW = grad.T @ self.x
        self.deltab = grad.sum(0)
        return grad @ self.W


class Sigmoid:
    def forward(self, x):
        self.a = 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
        return self.a

    def backward(self, grad):
        return grad * self.a * (1 - self.a)


class Identity:
    def forward(self, x):
        return x

    def backward(self, grad):
        return grad


class MLP:
    def __init__(self, blocks: list):
        self.blocks = blocks

    def forward(self, x):
        for block in self.blocks:
            x = block.forward(x)
        return x

    def backward(self, grad):
        for block in self.blocks[::-1]:
            grad = block.backward(grad)
        return grad


class MSELoss:
    def forward(self, pred, true):
        self.pred = pred
        self.true = true
        self.N = pred.shape[0]
        return ((pred - true) ** 2).mean()

    def backward(self):
        return (2 / self.N) * (self.pred - self.true)

    def __call__(self, pred, true):
        return self.forward(pred, true)


class SGDOptimizer:
    def __init__(self, lr, model):
        self.lr = lr
        self.model = model

    def step(self):
        for block in self.model.blocks:
            if block.__class__ == Linear:
                block.W = block.W - self.lr * block.deltaW
                block.b = block.b - self.lr * block.deltab


def train(model, optimizer, X_train, Y_train, X_test, Y_test,
          Y_mean, Y_std, loss_func=MSELoss(), epochs=5000):
    history = {"train_loss": [], "test_loss": [],
               "train_mae": [],  "test_mae": []}

    for epoch in range(epochs):
        prediction = model.forward(X_train)
        loss = loss_func(prediction, Y_train)
        grad = loss_func.backward()
        model.backward(grad)
        optimizer.step()

        history["train_loss"].append(loss)
        train_mae = np.abs(
            prediction * Y_std + Y_mean - (Y_train * Y_std + Y_mean)
        ).mean()
        history["train_mae"].append(train_mae)

        test_pred = model.forward(X_test)
        test_loss = ((test_pred - Y_test) ** 2).mean()
        test_mae = np.abs(
            test_pred * Y_std + Y_mean - (Y_test * Y_std + Y_mean)
        ).mean()
        history["test_loss"].append(test_loss)
        history["test_mae"].append(test_mae)

        if epoch % 500 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch:4d} | Train Loss: {loss:.6f} | Test Loss: {test_loss:.6f}")

    return history


def evaluate(model, X_test, Y_test, Y_mean, Y_std):
    Y_pred = model.forward(X_test) * Y_std + Y_mean
    Y_real = Y_test * Y_std + Y_mean
    mae = np.abs(Y_pred - Y_real).mean()
    print(f"TEST PREDICTIONS (UNSEEN DATA):")
    print(f"  MAE: £{mae:,.0f}")
    return mae


def visualization(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="Train Loss", color="#e74c3c", linewidth=1.5)
    ax1.plot(history["test_loss"],  label="Test Loss",  color="#3498db", linewidth=1.5)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("MSE Loss (normalized)")
    ax1.set_title("Training vs Test Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history["train_mae"], label="Train MAE", color="#e74c3c", linewidth=1.5)
    ax2.plot(history["test_mae"],  label="Test MAE",  color="#3498db", linewidth=1.5)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("MAE (£)")
    ax2.set_title("Training vs Test MAE")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


np.random.seed(15)
model_baseline = MLP([
    Linear(D, 64), Sigmoid(),
    Linear(64, 32), Sigmoid(),
    Linear(32, 1),  Identity()
])
optimizer_baseline = SGDOptimizer(lr=0.01, model=model_baseline)

print("=== Baseline: Sigmoid + SGD (5000 epochs) ===")
history_baseline = train(model_baseline, optimizer_baseline,
                         X_train, Y_train, X_test, Y_test, Y_mean, Y_std,
                         epochs=5000)
print()
mae_baseline = evaluate(model_baseline, X_test, Y_test, Y_mean, Y_std)
visualization(history_baseline)

## 2. Extension 1: ReLU + SGD

### 2.1 ReLU Activation

**Forward:**

$$\text{ReLU}(z) = \max(0, z)$$

**Backward** (given upstream gradient $\texttt{grad}$):

$$\frac{\partial \text{ReLU}}{\partial z} = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases} \qquad \Longrightarrow \qquad \frac{\partial \mathcal{L}}{\partial z} = \texttt{grad} \cdot \mathbf{1}_{z > 0}$$

Unlike Sigmoid whose derivative peaks at 0.25, ReLU passes the gradient through unchanged for any positive input. This means gradients do not shrink as they flow through multiple layers, solving the vanishing gradient problem.

In [ ]:
class ReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)

    def backward(self, grad):
        return grad * (self.x > 0)

### 2.2 Training with ReLU

In [ ]:
np.random.seed(15)
model_relu = MLP([
    Linear(D, 64), ReLU(),
    Linear(64, 32), ReLU(),
    Linear(32, 1),  Identity()
])
optimizer_relu = SGDOptimizer(lr=0.01, model=model_relu)

print("=== Extension 1: ReLU + SGD (5000 epochs) ===")
history_relu = train(model_relu, optimizer_relu,
                     X_train, Y_train, X_test, Y_test, Y_mean, Y_std,
                     epochs=5000)
print()
mae_relu = evaluate(model_relu, X_test, Y_test, Y_mean, Y_std)
visualization(history_relu)

## 3. Extension 2: ReLU + Adam

### 3.1 Adam Optimizer

Adam maintains per-parameter **first moment** (mean of gradients, like momentum) and **second moment** (mean of squared gradients, like RMSProp) estimates, with bias correction for the initial steps.

In [ ]:
class AdamOptimizer:
    def __init__(self, lr, model, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.model = model
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0

        self.state = []
        for block in self.model.blocks:
            if block.__class__ == Linear:
                self.state.append({
                    "layer": block,
                    "mW": np.zeros_like(block.W),
                    "vW": np.zeros_like(block.W),
                    "mb": np.zeros_like(block.b),
                    "vb": np.zeros_like(block.b),
                })

    def step(self):
        self.t += 1

        for s in self.state:
            layer = s["layer"]
            gW, gb = layer.deltaW, layer.deltab

            # 1st moment (momentum)
            s["mW"] = self.beta1 * s["mW"] + (1 - self.beta1) * gW
            s["mb"] = self.beta1 * s["mb"] + (1 - self.beta1) * gb

            # 2nd moment (adaptive learning rate)
            s["vW"] = self.beta2 * s["vW"] + (1 - self.beta2) * (gW ** 2)
            s["vb"] = self.beta2 * s["vb"] + (1 - self.beta2) * (gb ** 2)

            # Bias correction
            mW_hat = s["mW"] / (1 - self.beta1 ** self.t)
            mb_hat = s["mb"] / (1 - self.beta1 ** self.t)
            vW_hat = s["vW"] / (1 - self.beta2 ** self.t)
            vb_hat = s["vb"] / (1 - self.beta2 ** self.t)

            # Update
            layer.W = layer.W - self.lr * mW_hat / (np.sqrt(vW_hat) + self.eps)
            layer.b = layer.b - self.lr * mb_hat / (np.sqrt(vb_hat) + self.eps)

### 3.2 Training with ReLU + Adam (100 epochs only)

In [ ]:
np.random.seed(15)
model_adam = MLP([
    Linear(D, 64), ReLU(),
    Linear(64, 32), ReLU(),
    Linear(32, 1),  Identity()
])
optimizer_adam = AdamOptimizer(lr=0.01, model=model_adam)

print("=== Extension 2: ReLU + Adam (100 epochs) ===")
history_adam = train(model_adam, optimizer_adam,
                    X_train, Y_train, X_test, Y_test, Y_mean, Y_std,
                    epochs=100)
print()
mae_adam = evaluate(model_adam, X_test, Y_test, Y_mean, Y_std)
visualization(history_adam)

## 4. Comparison

In [ ]:
print(f"{'Model':<35s} {'Epochs':>8s} {'MAE':>15s}")
print(f"{'-'*58}")
print(f"{'Sigmoid + SGD (baseline)':<35s} {'5000':>8s} {'£'+f'{mae_baseline:,.0f}':>15s}")
print(f"{'ReLU + SGD':<35s} {'5000':>8s} {'£'+f'{mae_relu:,.0f}':>15s}")
print(f"{'ReLU + Adam':<35s} {'100':>8s} {'£'+f'{mae_adam:,.0f}':>15s}")

> #### 📝 Key Takeaways
>
> **ReLU vs Sigmoid**: Sigmoid's derivative peaks at 0.25, so gradients shrink by at least 75% at every hidden layer. With 2 hidden layers, the first layer receives at most $0.25^2 = 6.25\%$ of the original gradient. ReLU passes gradients through unchanged for positive inputs (derivative = 1), eliminating this bottleneck.
>
> **Adam vs SGD**: SGD uses the same learning rate for every parameter. Adam adapts the learning rate per-parameter based on the history of gradients, allowing it to take larger steps for infrequently updated parameters and smaller steps for frequently updated ones. The momentum component also smooths out noisy gradients. This is why Adam achieves in 100 epochs what SGD needs 5000 for.

## Summary

- Replacing **Sigmoid with ReLU** in hidden layers solves the vanishing gradient problem and improves both convergence speed and final accuracy, with no other code changes needed.
- Replacing **SGD with Adam** provides per-parameter adaptive learning rates and momentum, achieving comparable or better results in roughly 50x fewer epochs.
- Both improvements require only swapping one building block each. The `MLP` container, `MSELoss`, `train()` function, and `Linear` layer remain completely unchanged, demonstrating the value of the modular design.